# 04 · Scotiabank EES Desk: Quantitative Trading Developer
**Onsite: Mon Oct 26, 11:05–12:05 · 40 Temperance St, 5th floor** · business professional · arrive 10:50

An **Electronic Execution Services** desk runs the systems that execute orders electronically: execution algorithms (VWAP, TWAP, POV...), smart order routing across Canadian venues, direct market access, pre-trade risk checks and transaction-cost analysis. Your execution simulator + RL execution agent is **the same problem the desk solves**, so it is your main asset in this interview.

| Part | What | Priority | Time |
|---|---|---|---|
| 1 | How an execution desk works (the big picture) | 🔴 | 20 min |
| 2 | Market microstructure Q&A | 🔴 | 45 min |
| 3 | Execution algorithms + TCA (code) | 🔴 | 60 min |
| 4 | Smart order routing (code) | 🟠 | 30 min |
| 5 | FIX protocol (code) | 🟠 | 30 min |
| 6 | **Order book / matching engine (code)**: the likeliest coding question | 🔴 | 60 min, twice |
| 7 | Pre-trade risk gate (code) | 🟡 | 25 min |
| 8 | Systems Q&A for trading developers | 🟠 | 30 min |
| 9 | Your project, told for this desk | 🔴 | 45 min, out loud |
| 10 | Behavioral + "why" answers | 🔴 | 30 min, out loud |
| 11 | Questions to ask them | 🟠 | 10 min |
| 12 | Mock list: rapid fire with a timer | 🔴 | 45 min |

### Plan
| When | Do |
|---|---|
| **Sat night** | Part 1, Part 9 (write your pitch), Part 10. Say the Squarepoint trivia answers out loud once. |
| **Sun morning** | Parts 2, 3, 5 |
| **Sun afternoon** | Part 6 twice (second time on paper, 25-min timer). Parts 4, 7. |
| **Sun evening** | Part 8, Part 12 out loud, Part 11. Lay out clothes, print 3 résumés. Sleep by 11. |
| **Mon morning** | Skim ⭐ answers + your pitch. Leave with buffer. |

**Run the setup cell first.** Exercises print ✅/❌. Answers are collapsed: try first.

- Order book: two sorted lists. Buyers (“bids”, highest price first) and sellers (“asks”, lowest price first). When a buyer’s price is at or above a seller’s price, they trade. It’s a priority queue problem.
- Market order vs limit order: “buy now at whatever price” vs “buy only at $X or better”.
- Spread: the gap between the best sell price and the best buy price.
- Why split big orders: one huge order eats through many prices and pushes the price up, so you’d pay more. The desk’s software breaks it into small pieces over time. It’s like rate-limiting your own requests so you don’t overload a server.
- VWAP: the average price weighted by volume, sum(price × qty) / sum(qty). Algorithms try to match the day’s VWAP.
- What the desk builds: software that takes a client’s big order, splits it into small orders, sends them to exchanges, checks risk limits, and measures how well it went. That’s backend engineering: queues, low latency, reliability, testing.

In [ ]:
import copy, heapq, math
from collections import deque, defaultdict

def _approx(x):
    if isinstance(x, float): return round(x, 4)
    if isinstance(x, dict): return {k: _approx(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return type(x)(_approx(v) for v in x)
    return x

def check(fn, cases):
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*copy.deepcopy(args))
        except Exception as e:
            print(f"  ✗ case {i}: raised {type(e).__name__}: {e}"); continue
        if _approx(got) == _approx(expected): passed += 1
        else: print(f"  ✗ case {i}: args={args!r}\n      expected={expected!r}\n      got     ={got!r}")
    print(f"{'✅' if passed == len(cases) else '❌'} {passed}/{len(cases)} passed")
    return passed == len(cases)

def expect(label, got, expected):
    ok = _approx(got) == _approx(expected)
    print(("✅ " if ok else "❌ ") + label + ("" if ok else f"\n      expected={expected!r}\n      got     ={got!r}"))
    return ok

# ---- FIX helpers used in Part 5 (given, not an exercise) ----
SOH = "\x01"
def build_fix(msg_type, fields, begin="FIX.4.2"):
    """fields: list of (tag, value) AFTER 35. Adds 8, 9, 35 and 10 correctly."""
    body = f"35={msg_type}{SOH}" + "".join(f"{t}={v}{SOH}" for t, v in fields)
    head = f"8={begin}{SOH}9={len(body.encode())}{SOH}"
    cks = sum((head + body).encode()) % 256
    return head + body + f"10={cks:03d}{SOH}"
def show(msg): return msg.replace(SOH, " | ")

---
# 1 · How an execution desk works

```
 CLIENT (pension fund, asset manager, internal trader)
   │  "Buy 500,000 RY, VWAP, today, limit $150"      ← parent order, usually over FIX
   ▼
 OMS / EMS ── pre-trade risk checks (size, price band, credit, short-sell rules, kill switch)
   ▼
 ALGO ENGINE ── decides WHEN and HOW MUCH: slices the parent into child orders
   │            (VWAP / TWAP / POV / Implementation Shortfall / Iceberg / Close)
   ▼
 SMART ORDER ROUTER ── decides WHERE: which venue(s), at what price, lit or dark
   ▼
 MARKET GATEWAYS ── speak each venue's protocol (FIX / binary)
   ▼
 VENUES: TSX, TSX Alpha, Nasdaq CXC/CX2, Cboe Canada, CSE, dark pools (e.g. MatchNow) ...
   │
   └── fills (ExecutionReports) flow back up: router → algo → client
 POST-TRADE: TCA (how good was the execution?), allocations, reporting
```
**Who the desk serves:** institutional clients who want to trade large orders without moving the price. The desk earns commissions, so **execution quality is the product**.

**What a trading developer on the desk builds:** algo logic and parameters, the router, market-data handlers, risk checks, client FIX connectivity, TCA tools and dashboards, simulation/back-testing of algos, and production support during market hours (09:30–16:00 ET).

**Why it's hard:** markets are fragmented across many venues, latency matters, mistakes cost real money instantly, regulation (best execution, order protection) applies, and the systems must never send runaway orders. Knight Capital lost ~$440M in 45 minutes in 2012 from a bad deployment; it's a good example to mention when talking about testing and kill switches.

> Venue names and the exact desk setup are from public knowledge and may be out of date; use them as vocabulary, and ask the interviewers how *their* stack looks.

---
# 2 · Market microstructure
Short, sayable answers. ⭐ = very likely.

**Q1. ⭐ What is a limit order book? Bid, ask, spread, mid?**

<details><summary>Answer</summary>

The list of resting **limit orders** at each price. **Bids** = buyers, **asks/offers** = sellers. **Best bid** = highest bid, **best ask** = lowest ask (together the *top of book* / *BBO*). **Spread** = ask − bid, the cost of crossing immediately. **Mid** = (bid + ask)/2, the usual "fair price" reference. Depth = total quantity at each level.
</details>

**Q2. ⭐ Market order vs limit order? Name other order types.**

<details><summary>Answer</summary>

**Market:** execute now at the best available prices. Certain execution, uncertain price (pays the spread, can walk the book). **Limit:** buy no higher / sell no lower than a price. Certain price, uncertain execution.
Others: **Stop** (becomes market when a trigger price trades), **IOC** (immediate-or-cancel: fill what you can now, cancel the rest), **FOK** (fill-or-kill: all now or nothing), **Day/GTC** (time in force), **Iceberg/reserve** (shows only part of the size), **Post-only** (only rest, never take liquidity), **MOC/LOC** (market/limit-on-close, executes in the closing auction).
</details>

**Q3. ⭐ What is price-time priority?**

<details><summary>Answer</summary>

How most lit venues match orders: **best price first**; at the same price, **first come first served** (FIFO). So your queue position matters: an order that joins a crowded level late may never fill. Replacing an order to change price or increase size usually **loses your time priority**; decreasing size usually keeps it.
</details>

**Q4. ⭐ Maker vs taker? What is maker-taker pricing?**

<details><summary>Answer</summary>

A **maker** adds liquidity (a resting limit order that doesn't cross). A **taker** removes it (a marketable order that trades immediately). Under **maker-taker** fees, venues pay makers a small rebate and charge takers a fee (fractions of a cent per share). **Inverted** venues do the reverse. Fees matter to the router: the best *displayed* price isn't always the best price *after fees*.
</details>

**Q5. ⭐ What is slippage? What is market impact?**

<details><summary>Answer</summary>

**Slippage:** the difference between the price you expected (decision/arrival price) and the price you actually got.
**Market impact:** the price move *caused by your own trading*. Buying pushes the price up. **Temporary impact** (from consuming liquidity) fades after you stop; **permanent impact** (information leaked to the market) stays. A common rule of thumb: impact grows roughly with the **square root** of order size relative to daily volume.
Trade-off: trade fast → more impact; trade slowly → more **timing risk** (the price drifts while you wait). Execution algorithms balance these two.
</details>

**Q6. ⭐ Why not just send a 500,000-share order to the exchange at once?**

<details><summary>Answer</summary>

It would sweep through many price levels, so you'd pay much worse prices (impact). It would also **signal** your intent, and others would trade ahead of you. So the desk splits it into many small **child orders** over time and across venues, using limit orders where possible to earn the spread rather than pay it.
</details>

**Q7. What is adverse selection?**

<details><summary>Answer</summary>

When your resting order gets filled mainly *because* the price is about to move against you. Example: your bid fills just before the price drops, because the seller knew something. Fills that "look good" can be costly. Market makers and passive algorithms manage it by watching short-term signals, like book imbalance, and pulling quotes.
</details>

**Q8. What is liquidity? How do you measure it?**

<details><summary>Answer</summary>

How easily you can trade size without moving the price. Measures: spread width, depth at top levels, average daily volume (ADV), and impact per share traded. A liquid stock (e.g. a big Canadian bank) has tight spreads and deep books; an illiquid small-cap needs much more care.
</details>

**Q9. What is tick size, and why does it matter?**

<details><summary>Answer</summary>

The minimum price increment (e.g. $0.01 for most stocks above $0.50 in Canada). Tick size shapes behaviour: if the spread is one tick wide, you can't improve the price, so queue position decides who fills. A wide tick-to-price ratio makes queues long and passive fills slow.
</details>

**Q10. What are the opening and closing auctions? Why is the close important?**

<details><summary>Answer</summary>

Instead of continuous matching, orders are collected and matched at one price that maximizes traded volume. The **closing auction** is a big share of daily volume (index funds and benchmarked funds trade at the close), so desks offer **MOC / at-close** algorithms, and the closing price is the benchmark for many clients.
</details>

**Q11. ⭐ ⭐ Canada specifics: fragmentation, Order Protection Rule, dark pools**

<details><summary>Answer</summary>

Canadian equities trade on many **marketplaces**: TSX, TSX Alpha, Nasdaq CXC/CX2, Cboe Canada, CSE and others, plus **dark pools** that don't display quotes (e.g. MatchNow).
The **Order Protection Rule (OPR)** requires that better-priced *visible* orders on any protected marketplace are filled before trading at a worse price elsewhere (no "trade-throughs"). So the router must check all protected venues' prices. That's why routers matter in Canada.
**Dark pools** let large orders trade without showing size (less information leakage), but fills are uncertain.
*(Say "as far as I know" on venue specifics. Rules and venues change.)*
</details>

**Q12. What is best execution?**

<details><summary>Answer</summary>

The duty to get clients the best overall result, considering price, speed, likelihood of execution, costs and the client's instructions. It isn't just "best price right now". Desks show they meet it with TCA reports.
</details>

**Q13. What is HFT, and how does it relate to an execution desk?**

<details><summary>Answer</summary>

High-frequency trading firms use very fast systems to make markets or arbitrage tiny price differences. For an execution desk they're both a liquidity source (tight spreads) and a risk: they may detect a predictable algo and trade ahead of it. So good algos randomize order size and timing to avoid being predictable.
</details>

---
# 3 · Execution algorithms + TCA

**Q14. ⭐ ⭐ Explain TWAP, VWAP, POV, Implementation Shortfall in one breath each.**

<details><summary>Answer</summary>

- **TWAP (time-weighted average price):** split the order evenly across time buckets. Simple, predictable (which also makes it detectable).
- **VWAP (volume-weighted average price):** trade in proportion to the market's **typical volume curve** (a U-shape: busy at the open and close, quiet midday), so your average price tracks the day's VWAP. The schedule comes from historical volume profiles.
- **POV / participation (e.g. 10%):** trade a fixed share of *actual* volume as it happens. Adapts to the day, but finish time is uncertain.
- **Implementation Shortfall (arrival price):** minimize cost vs the price when the order arrived. Usually trades faster at the start to reduce timing risk, then slows down. Balances impact against risk (the **Almgren–Chriss** model formalizes this).
- Also: **Iceberg** (show a small slice at a time), **Close/MOC** (target the closing price), **Liquidity-seeking** (hunt for size across lit and dark venues).
</details>

**Q15. ⭐ What is the VWAP formula?**

<details><summary>Answer</summary>

VWAP = Σ(priceᵢ × qtyᵢ) / Σ qtyᵢ over all trades in the period. *Never* the plain average of prices.
</details>

**Q16. ⭐ ⭐ What is TCA? How do you measure whether an execution was good?**

<details><summary>Answer</summary>

**Transaction Cost Analysis.** Compare your average fill price to benchmarks:
- **vs arrival price** (implementation shortfall): what it cost compared with trading at the moment the order arrived.
- **vs interval VWAP:** did you beat the market's average over the period?
- **vs close**, for close-benchmarked orders.
Express results in **basis points** (1 bp = 0.01%). For a **buy**, slippage (bps) = (avg_fill − benchmark) / benchmark × 10,000, so **positive = worse**. For a sell, flip the sign.
Also look at **fill rate**, **market impact** (price path during and after the order), **reversion** (price coming back after you finish = you pushed it), and **venue fill quality**.
</details>

**Q17. ⭐ How do you test or back-test an execution algorithm?**

<details><summary>Answer</summary>

Replay historical market data (full order book, not just trades) through a **simulator** that models your child orders' queue position, fills, latency and fees, then compute TCA vs benchmarks over many orders. The hard part: real orders change the market (impact), which a replay can't capture fully, so you add an impact model and validate with small live trials (A/B tests between algo versions). *This is exactly your project. Say so.*
</details>

## ✍️ 3a · TWAP and VWAP schedules
Return integer child quantities per bucket that **sum exactly to `total`**.
- `twap_schedule(total, n)`: as even as possible.
- `vwap_schedule(total, profile)`: proportional to `profile` (fractions that sum to 1).
Use **largest-remainder rounding**: floor each share, then give the leftover shares one at a time to the buckets with the largest fractional parts (ties: earlier bucket first).

In [ ]:
PROFILE = [0.13, 0.09, 0.07, 0.06, 0.05, 0.05, 0.05, 0.05, 0.06, 0.07, 0.08, 0.10, 0.14]  # 13 half-hours, U-shape

def twap_schedule(total: int, n: int) -> list[int]:
    pass

def vwap_schedule(total: int, profile: list[float]) -> list[int]:
    pass

# --- tests ---
check(twap_schedule, [((10, 3), [4, 3, 3]), ((9, 3), [3, 3, 3]), ((2, 4), [1, 1, 0, 0])])
check(vwap_schedule, [((100, [0.5, 0.25, 0.25]), [50, 25, 25]), ((10, [0.33, 0.33, 0.34]), [3, 3, 4]),
                      ((100_000, PROFILE), [13000, 9000, 7000, 6000, 5000, 5000, 5000, 5000, 6000, 7000, 8000, 10000, 14000])])
print("shares conserved:", sum(vwap_schedule(123_457, PROFILE) or []) == 123_457)

<details><summary>💡 <b>Hint</b></summary>

TWAP is just VWAP with a flat profile `[1/n]*n`. Compute `raw = total * w`, `base = floor(raw)`, `left = total - sum(base)`, then sort bucket indexes by `(-(raw - base), index)` and add 1 to the first `left` of them.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
PROFILE = [0.13, 0.09, 0.07, 0.06, 0.05, 0.05, 0.05, 0.05, 0.06, 0.07, 0.08, 0.10, 0.14]

def vwap_schedule(total: int, profile: list[float]) -> list[int]:
    raw = [total * w for w in profile]
    base = [int(math.floor(x + 1e-9)) for x in raw]
    left = total - sum(base)
    order = sorted(range(len(raw)), key=lambda i: (-(raw[i] - base[i]), i))
    for i in order[:left]:
        base[i] += 1
    return base

def twap_schedule(total: int, n: int) -> list[int]:
    return vwap_schedule(total, [1 / n] * n)
```
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Rounding must **conserve shares**: a schedule that sums to 99,999 of a 100,000-share order is a bug a trader will notice. In a real algo the schedule is only a target: the engine checks "am I ahead or behind?" each interval and adjusts, with limits on how far it may deviate.
</details>

## ✍️ 3b · POV (participation) slicing
`pov_child_orders(total, market_volumes, rate)`: in each interval the market trades `market_volumes[i]` shares; trade `floor(rate × volume)` but never more than what's left. Return the list of child quantities (stop adding entries once the order is complete).

In [ ]:
def pov_child_orders(total: int, market_volumes: list[int], rate: float) -> list[int]:
    pass

# --- tests ---
check(pov_child_orders, [((1000, [2000, 5000, 3000, 8000], 0.1), [200, 500, 300]),
                          ((1000, [2000, 3000], 0.1), [200, 300]),
                          ((50, [100, 1000], 0.2), [20, 30])])

<details><summary>💡 <b>Hint</b></summary>

Track `remaining`; for each volume, `q = min(remaining, int(rate * v))`; append; stop when `remaining == 0`.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def pov_child_orders(total: int, market_volumes: list[int], rate: float) -> list[int]:
    out, remaining = [], total
    for v in market_volumes:
        if remaining == 0:
            break
        q = min(remaining, int(rate * v + 1e-9))
        out.append(q)
        remaining -= q
    return out
```
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

POV adapts to real volume (a busy day finishes sooner), but **completion is not guaranteed**: case 2 ends with 500 shares unfilled. Real POV algos add min/max rates, a deadline, and avoid counting their own fills as "market volume".
</details>

## ✍️ 3c · TCA report
`tca(side, fills, arrival_px, market_trades)`. `fills` and `market_trades` are lists of `(price, qty)`. Return a dict with:
- `avg_px`: your volume-weighted average fill price
- `filled`: total quantity
- `mkt_vwap`: VWAP of the market trades
- `slip_arrival_bps`, `slip_vwap_bps`: cost in bps vs each benchmark, **positive = worse for the client** (buy: paid more; sell: received less). Round all floats to 2 dp.

In [ ]:
def tca(side: str, fills, arrival_px: float, market_trades) -> dict:
    pass

# --- tests ---
check(tca, [(("BUY", [(100.0, 300), (100.2, 200)], 100.0, [(99.9, 1000), (100.1, 1000), (100.3, 2000)]),
               {"avg_px": 100.08, "filled": 500, "mkt_vwap": 100.15, "slip_arrival_bps": 8.0, "slip_vwap_bps": -6.99}),
              (("SELL", [(50.0, 100)], 50.5, [(50.0, 100), (51.0, 100)]),
               {"avg_px": 50.0, "filled": 100, "mkt_vwap": 50.5, "slip_arrival_bps": 99.01, "slip_vwap_bps": 99.01})])

<details><summary>💡 <b>Hint</b></summary>

VWAP helper: `sum(p*q)/sum(q)`. `sign = 1 if BUY else -1`. `bps = sign * (avg - bench) / bench * 1e4`.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def tca(side: str, fills, arrival_px: float, market_trades) -> dict:
    vwap = lambda xs: sum(p * q for p, q in xs) / sum(q for _, q in xs)
    avg = vwap(fills)
    mv = vwap(market_trades)
    sign = 1 if side == "BUY" else -1
    bps = lambda bench: round(sign * (avg - bench) / bench * 1e4, 2)
    return {"avg_px": round(avg, 2), "filled": sum(q for _, q in fills), "mkt_vwap": round(mv, 2),
            "slip_arrival_bps": bps(arrival_px), "slip_vwap_bps": bps(mv)}
```
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Know the **sign convention** cold and say it: "positive means it cost the client". In case 1 the buy was **8 bps worse than arrival but 7 bps better than VWAP**: the market rallied during the order. Which benchmark matters depends on the client's instruction, and that is exactly the kind of nuance interviewers like to hear.
</details>

---
# 4 · Smart order routing

**Q18. ⭐ What does a smart order router do?**

<details><summary>Answer</summary>

Given a child order, it decides **where** to send it: split across venues by available price and size, after **fees/rebates**, respecting **order protection**, using **historical fill rates** per venue (displayed size may vanish before you arrive), and possibly checking dark pools first. It also handles partial fills and re-routes the remainder.
</details>

## ✍️ 4a · Route a marketable buy across venues
Each venue has an ask book `[(price, qty), ...]` (best first) and a **taker fee per share**. Fill `qty` shares by always taking the level with the lowest **effective price = price + fee** (ties: the venue name that sorts first). Return `(allocations, avg_effective_px)` where `allocations = {venue: shares}` (only venues used) and the average is rounded to 4 dp. If total liquidity is short, fill what you can.

In [ ]:
def route_buy(qty: int, books: dict, fees: dict):
    pass

# --- tests ---
books = {"TSX": [(10.00, 300), (10.01, 500)], "CXC": [(10.00, 200), (10.02, 1000)], "ALPHA": [(9.995, 100)]}
fees  = {"TSX": 0.0030, "CXC": 0.0010, "ALPHA": 0.0100}
check(route_buy, [((400, books, fees), ({"CXC": 200, "TSX": 200}, 10.002)),
                  ((700, books, fees), ({"CXC": 200, "TSX": 400, "ALPHA": 100}, 10.0041)),
                  ((5000, books, fees), ({"CXC": 1200, "TSX": 800, "ALPHA": 100}, 10.0139))])

<details><summary>💡 <b>Hint</b></summary>

Push every level onto a min-heap keyed by `(price + fee, venue, level_index)`, or push only each venue's next level as you consume (k-way merge). Pop and take `min(remaining, level_qty)`.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def route_buy(qty: int, books: dict, fees: dict):
    heap = []
    for v, levels in books.items():
        if levels:
            p, q = levels[0]
            heapq.heappush(heap, (p + fees[v], v, 0))
    alloc, cost, remaining = {}, 0.0, qty
    while remaining and heap:
        eff, v, i = heapq.heappop(heap)
        take = min(remaining, books[v][i][1])
        alloc[v] = alloc.get(v, 0) + take
        cost += take * eff
        remaining -= take
        if i + 1 < len(books[v]):                       # next level of the same venue
            p, q = books[v][i + 1]
            heapq.heappush(heap, (p + fees[v], v, i + 1))
    filled = qty - remaining
    return alloc, round(cost / filled, 4) if filled else 0.0
```
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

**k-way merge with a heap**, the same pattern as merging k sorted lists. The interesting result: ALPHA shows the best *raw* price (9.995) but is the *worst after fees* (10.005). Follow-ups to mention: displayed size may disappear (so you'd weight venues by fill probability), latency differences between venues (send to slower venues first so all orders arrive together), and Order Protection Rule constraints.
</details>

---
# 5 · FIX protocol
FIX = the standard text protocol banks, clients and venues use to send orders. Messages are `tag=value` pairs separated by the SOH character (`\x01`, shown here as ` | `).

| Tag | Name | Example | | Tag | Name | Example |
|---|---|---|---|---|---|---|
| 8 | BeginString | FIX.4.2 | | 11 | ClOrdID (client order id) | ORD123 |
| 9 | BodyLength | 112 | | 55 | Symbol | RY |
| **35** | **MsgType** | D | | 54 | Side | 1 = Buy, 2 = Sell |
| 49 / 56 | Sender / Target CompID | CLIENT / SCOTIA | | 38 | OrderQty | 500 |
| 34 | MsgSeqNum | 42 | | 40 | OrdType | 1 = Market, 2 = Limit |
| 52 | SendingTime | 20261026-15:05:00 | | 44 | Price | 150.25 |
| **10** | **CheckSum** (last field) | 087 | | 59 | TimeInForce | 0 = Day, 3 = IOC |
| 39 | OrdStatus | 0 new, 1 partial, 2 filled, 4 cancelled, 8 rejected | | 150 | ExecType | 0 new, F trade (4.4), 4 cancelled |
| 31 / 32 | LastPx / LastQty | fill price / size | | 14 / 151 | CumQty / LeavesQty | filled so far / remaining |

**Message types (35=):** `D` NewOrderSingle · `8` ExecutionReport · `F` OrderCancelRequest · `G` OrderCancelReplaceRequest · `9` OrderCancelReject · `A` Logon · `0` Heartbeat · `5` Logout · `2` ResendRequest · `3` Reject.

**A typical flow:** client sends `D` → desk replies `8` with 39=0 (New) → fills arrive as `8` with 39=1 (Partially filled) / 39=2 (Filled) → a cancel is `F`, answered by `8` with 39=4 or a `9` reject.

**Session layer facts to know:** every message has a **sequence number (34)**; a gap means messages were missed, so you send a **ResendRequest (35=2)**. **Heartbeats** detect dead connections. **CheckSum (10)** = sum of all bytes before the `10=` field, mod 256, as 3 digits. **BodyLength (9)** = bytes from after the `9=...|` field up to just before `10=`.

In [ ]:
msg = build_fix("D", [(49, "CLIENT"), (56, "SCOTIA"), (34, 42), (11, "ORD123"), (55, "RY"), (54, 1), (38, 500), (40, 2), (44, "150.25"), (59, 0)])
print(show(msg))

**Q19. ⭐ Why does ClOrdID matter? What if the same order is sent twice?**

<details><summary>Answer</summary>

ClOrdID uniquely identifies an order from the sender, so the receiver can **reject duplicates** (idempotency) after reconnects/resends. Cancel and replace messages reference the original via **OrigClOrdID (41)**. Without this, a network retry could double an order, which is a real-money bug.
</details>

## ✍️ 5a · Parse and validate a FIX message
`parse_fix(msg)` → dict `{tag(int): value(str)}`. Raise `ValueError` if the **checksum** or **body length** is wrong, or if the first three tags are not 8, 9, 35 in order.

In [ ]:
def parse_fix(msg: str) -> dict:
    pass

good = build_fix("D", [(11, "ORD123"), (55, "RY"), (54, 1), (38, 500), (40, 2), (44, "150.25")])
bad_ck = good[:-4] + "000" + SOH
bad_len = good.replace("38=500", "38=5000")      # body changed, header not updated

# --- tests ---
d = parse_fix(good) or {}
expect("fields parsed", (d.get(35), d.get(55), d.get(38), d.get(44)), ("D", "RY", "500", "150.25"))
for label, m in [("bad checksum rejected", bad_ck), ("bad body length rejected", bad_len)]:
    try:
        parse_fix(m); expect(label, "no error", "ValueError")
    except ValueError:
        expect(label, True, True)

<details><summary>💡 <b>Hint</b></summary>

Split on SOH (drop the trailing empty string), split each field on the **first** `=`. Checksum: `sum(msg[:msg.rindex(SOH + "10=") + 1].encode()) % 256`. Body length: the bytes between the end of the `9=...SOH` field and the start of `10=`.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def parse_fix(msg: str) -> dict:
    parts = msg.split(SOH)
    if parts[-1] != "":
        raise ValueError("message must end with SOH")
    pairs = [p.split("=", 1) for p in parts[:-1]]
    if [t for t, _ in pairs[:3]] != ["8", "9", "35"] or pairs[-1][0] != "10":
        raise ValueError("bad header/trailer order")
    ck_start = msg.rindex(SOH + "10=") + 1                 # index where "10=" begins
    if sum(msg[:ck_start].encode()) % 256 != int(pairs[-1][1]):
        raise ValueError("bad checksum")
    body_start = msg.index(SOH, msg.index("9=")) + 1      # after the 9= field
    if len(msg[body_start:ck_start].encode()) != int(pairs[1][1]):
        raise ValueError("bad body length")
    return {int(t): v for t, v in pairs}
```
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Validation is cheap compared with acting on a corrupted order. In production you'd use a FIX engine (QuickFIX/J, QuickFIX/n, or an in-house C++ engine) rather than hand-parsing, but interviewers like seeing that you know what the engine checks: **header order, length, checksum, sequence numbers, duplicate ClOrdIDs**.
</details>

---
# 6 · Order book / matching engine (the likeliest coding question)
You built one from ITCH, so this is your home ground. Practise it twice: once here, once on paper with a 25-minute timer while talking out loud.

## ✍️ 6a · Limit order book with matching
Implement `OrderBook`:
- `add(oid, side, price, qty)`: side is `"B"` or `"S"`. Match against the opposite side **while prices cross**, best price first, FIFO within a price (**price-time priority**). Trades execute at the **resting** order's price. Any remainder rests on the book. Return a list of trades `(resting_oid, incoming_oid, price, qty)`.
- `cancel(oid)` → `True` if the order was live and is now removed, else `False`.
- `best_bid()`, `best_ask()` → `(price, total_qty)` or `None`.

Clarifying questions you'd ask first: duplicate ids? zero/negative qty? market orders? (Not required here.)

In [ ]:
class OrderBook:
    def __init__(self):
        pass
    def add(self, oid, side, price, qty):
        pass
    def cancel(self, oid):
        pass
    def best_bid(self):
        pass
    def best_ask(self):
        pass

# --- tests ---
ob = OrderBook()
expect("rest bids/asks", (ob.add(1, "B", 100, 10), ob.add(2, "B", 100, 5), ob.add(3, "B", 99, 7), ob.add(4, "S", 102, 8)), ([], [], [], []))
expect("top of book", (ob.best_bid(), ob.best_ask()), ((100, 15), (102, 8)))
expect("sell crosses: FIFO at 100, then 99", ob.add(5, "S", 99, 18), [(1, 5, 100, 10), (2, 5, 100, 5), (3, 5, 99, 3)])
expect("after sweep", (ob.best_bid(), ob.best_ask()), ((99, 4), (102, 8)))
expect("cancel live / dead", (ob.cancel(3), ob.cancel(3), ob.cancel(1)), (True, False, False))
expect("bid side empty", ob.best_bid(), None)
expect("buy above ask: trades at resting price, remainder rests", ob.add(6, "B", 105, 10), [(4, 6, 102, 8)])
expect("remainder", (ob.best_bid(), ob.best_ask()), ((105, 2), None))
ob.add(7, "S", 110, 5); ob.add(8, "S", 110, 5); ob.cancel(7)
expect("cancelled order is skipped in matching", ob.add(9, "B", 110, 6), [(8, 9, 110, 5)])
expect("final", (ob.best_bid(), ob.best_ask()), ((110, 1), None))

<details><summary>💡 <b>Hint</b></summary>

Data structures:
- `orders: dict oid → [side, price, qty]` for O(1) cancel lookup.
- `levels[side]: dict price → deque of oids` (FIFO queue per price) and `qty_at[side][price]` totals.
- Best price: a **heap per side** (bids as negative prices) with **lazy deletion**: when the top level is empty, pop it.
Cancel: set the order's qty to 0 and drop it from the dict; the matching loop skips dead ids when it meets them (lazy), or remove it from a linked list for true O(1).
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
class OrderBook:
    def __init__(self):
        self.orders = {}                                    # oid -> [side, price, qty]
        self.levels = {"B": defaultdict(deque), "S": defaultdict(deque)}
        self.qty_at = {"B": defaultdict(int), "S": defaultdict(int)}
        self.heaps = {"B": [], "S": []}                     # bids store -price

    def _push_level(self, side, price):
        heapq.heappush(self.heaps[side], -price if side == "B" else price)

    def _best(self, side):
        h = self.heaps[side]
        while h:
            p = -h[0] if side == "B" else h[0]
            if self.qty_at[side][p] > 0:
                return p
            heapq.heappop(h)                                # lazy delete empty level
            self.levels[side].pop(p, None); self.qty_at[side].pop(p, None)
        return None

    def add(self, oid, side, price, qty):
        opp = "S" if side == "B" else "B"
        crosses = (lambda p: p <= price) if side == "B" else (lambda p: p >= price)
        trades = []
        while qty > 0:
            best = self._best(opp)
            if best is None or not crosses(best):
                break
            q = self.levels[opp][best]
            while q and qty > 0:
                rid = q[0]
                rest = self.orders.get(rid)
                if rest is None:                            # cancelled earlier: skip
                    q.popleft(); continue
                fill = min(qty, rest[2])
                trades.append((rid, oid, best, fill))
                rest[2] -= fill; qty -= fill; self.qty_at[opp][best] -= fill
                if rest[2] == 0:
                    q.popleft(); del self.orders[rid]
        if qty > 0:                                         # rest the remainder
            self.orders[oid] = [side, price, qty]
            if self.qty_at[side][price] == 0:                # new (or emptied) level
                self._push_level(side, price)
            self.levels[side][price].append(oid)
            self.qty_at[side][price] += qty
        return trades

    def cancel(self, oid):
        o = self.orders.pop(oid, None)
        if o is None:
            return False
        side, price, qty = o
        self.qty_at[side][price] -= qty                     # oid stays in the deque; skipped lazily
        return True

    def best_bid(self):
        p = self._best("B")
        return None if p is None else (p, self.qty_at["B"][p])

    def best_ask(self):
        p = self._best("S")
        return None if p is None else (p, self.qty_at["S"][p])
```
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Say the complexity out loud: add = O(log L) for a new level + O(1) per fill; cancel = O(1) (lazy); best = amortized O(1). Then the C++ production version, which is what this desk cares about:
- **Prices as integer ticks** (never floats), price levels in a `std::map` or a **flat array indexed by tick** near the mid.
- **Intrusive doubly linked list** per level so cancel is truly O(1) without lazy garbage.
- **Pre-allocated order pool** (no `new` on the hot path), `unordered_map` (or a flat hash map) from id → order.
- Single-threaded matching per symbol (deterministic), fed by a lock-free queue.
Mention the ITCH message types you handled: Add, Execute, Cancel (partial), Delete, Replace. Replace = cancel + add, **losing time priority**.
</details>

---
# 7 · Pre-trade risk gate

**Q20. ⭐ ⭐ What pre-trade risk checks would you put in front of an algo? What is a kill switch?**

<details><summary>Answer</summary>

Checks on **every** order before it leaves: **max order size** and **max notional**, **price band** (reject prices far from the reference, a "fat-finger" check), **order-rate limits** (messages per second), **position/credit limits** per client, **duplicate order** detection, restricted/short-sell lists. A **kill switch** instantly stops sending and cancels all open orders, for a client, an algo, or the whole desk. These checks must be fast (on the hot path) and are required by regulation.
</details>

## ✍️ 7a · RiskGate
`RiskGate(max_qty, max_notional, band_pct, max_per_sec)`. `check(ts, qty, price, ref_price)` returns `"OK"` or the **first** failing reason in this order: `"QTY"`, `"NOTIONAL"`, `"PRICE_BAND"` (|price − ref| / ref > band_pct), `"RATE"` (more than `max_per_sec` **accepted** orders within the last 1.0 s, i.e. timestamps in `(ts − 1, ts]`). Only accepted orders count toward the rate.

In [ ]:
class RiskGate:
    def __init__(self, max_qty, max_notional, band_pct, max_per_sec):
        pass
    def check(self, ts, qty, price, ref_price):
        pass

# --- tests ---
g = RiskGate(max_qty=1000, max_notional=50_000, band_pct=0.05, max_per_sec=3)
res = [g.check(0.0, 100, 100, 100), g.check(0.1, 2000, 100, 100), g.check(0.2, 600, 100, 100),
       g.check(0.3, 10, 106, 100), g.check(0.4, 10, 100, 100), g.check(0.5, 10, 100, 100),
       g.check(0.9, 10, 100, 100), g.check(1.05, 10, 100, 100), g.check(1.45, 10, 100, 100)]
expect("risk results", res, ["OK", "QTY", "NOTIONAL", "PRICE_BAND", "OK", "OK", "RATE", "OK", "OK"])

<details><summary>💡 <b>Hint</b></summary>

Keep a `deque` of accepted timestamps. Before the rate check, pop from the left while `dq[0] <= ts - 1.0`. If `len(dq) >= max_per_sec` → RATE. Otherwise append ts and return OK.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
class RiskGate:
    def __init__(self, max_qty, max_notional, band_pct, max_per_sec):
        self.max_qty, self.max_notional = max_qty, max_notional
        self.band, self.max_per_sec = band_pct, max_per_sec
        self.accepted = deque()

    def check(self, ts, qty, price, ref_price):
        if qty > self.max_qty:
            return "QTY"
        if qty * price > self.max_notional:
            return "NOTIONAL"
        if abs(price - ref_price) / ref_price > self.band:
            return "PRICE_BAND"
        while self.accepted and self.accepted[0] <= ts - 1.0:
            self.accepted.popleft()
        if len(self.accepted) >= self.max_per_sec:
            return "RATE"
        self.accepted.append(ts)
        return "OK"
```
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Sliding-window rate limiter with a deque: O(1) amortized. Order the checks **cheapest and most important first**. Follow-ups: a token bucket (allows bursts), per-client limits, and the fact that real gates sit in front of *everything*, including the algos you trust.
</details>

---
# 8 · Systems questions for trading developers

**Q21. ⭐ ⭐ What does low latency mean for an execution desk, and how do you get it?**

<details><summary>Answer</summary>

Reacting to market data and sending orders in microseconds to milliseconds. For an *agency execution* desk the goal is usually **consistent, low-jitter** latency rather than HFT nanoseconds: a slow reaction means your child order arrives after the liquidity is gone.
How: C++ (or tuned Java) on the hot path, no heap allocation or locks in the critical loop, pre-allocated memory, pinned threads, binary protocols, efficient market-data handlers, co-location near exchanges, kernel-bypass networking. **Measure first** (timestamps at each hop, percentiles like p99, not averages).
</details>

**Q22. ⭐ Java vs C++ vs Python on a trading desk?**

<details><summary>Answer</summary>

**C++:** the hot path (market data, gateways, matching, latency-sensitive algos): full control, no GC. **Java:** very common for OMS/EMS and algo engines in banks: fast enough when tuned (avoid GC pressure), productive, safe. **Python:** research, TCA, analytics, tooling, testing, dashboards. Many desks mix all three. Ask them which they use.
</details>

**Q23. ⭐ How would you make sure a new algo version doesn't blow up in production?**

<details><summary>Answer</summary>

Unit tests on logic; **replay tests** on recorded market days; a simulator/certification environment with the venues; **deterministic** behaviour so a replay reproduces bugs; **risk gates and kill switch**; staged rollout (one client or a small % of flow first) with **feature flags**; monitoring and alerts on fill rate, slippage and reject counts; and a quick rollback. Mention Knight Capital (2012) as what happens without this.
</details>

**Q24. A venue connection drops mid-day. What should happen?**

<details><summary>Answer</summary>

Detect it (missed heartbeats); stop routing new orders to that venue; work out the state of orders that were live there (they may still be live or cancelled; reconcile on reconnect via **sequence numbers and resend**); re-route remaining quantity to other venues; alert support; never assume an order is dead without confirmation, to avoid **double fills**.
</details>

**Q25. How do you handle market data gaps?**

<details><summary>Answer</summary>

Feeds carry **sequence numbers**. On a gap: mark the book as stale (don't trade on it), request a retransmission or snapshot, rebuild, then resume. Trading on a wrong book is worse than pausing.
</details>

**Q26. ⭐ Why are prices often stored as integers?**

<details><summary>Answer</summary>

Floating point can't represent most decimals exactly (0.1 + 0.2 ≠ 0.3), so comparisons and sums drift. Exchanges and protocols like ITCH send prices as **fixed-point integers** (e.g. price × 10,000). Use integer ticks internally and convert only for display. You did this in your ITCH parser.
</details>

**Q27. Concurrency: how would you structure an algo engine?**

<details><summary>Answer</summary>

Common design: a market-data thread per feed → **lock-free queues** → strategy/algo threads, each owning its own orders (no shared mutable state, so no locks), → an order gateway thread. Each symbol is processed by one thread, so events stay in order and behaviour is deterministic. Use locks only off the hot path.
</details>

**Q28. ⭐ Squarepoint repeats: be ready to answer these in 60 seconds**

<details><summary>Answer</summary>

- What happens when you type a URL? DNS → TCP → TLS → HTTP → render.
- Docker, and what came before it: VMs, config management, virtualenv/chroot. A VM virtualises hardware; a container virtualises the OS.
- Generators: `yield` pauses with state kept; lazy, O(1) memory.
- Complexity: describes growth, not absolute time; constants, cache, best/avg/worst/amortized.
- Recursion: base case + smaller subproblem, O(depth) stack, convertible to iteration.
Full answers are in the previous chat; more in notebook 03.
</details>

---
# 9 · Your project, told for this desk
This is the part that wins the interview. Write your answers in the cells, then **say them out loud** with a timer.

### The 90-second version (structure)
1. **Problem:** "Executing a large order without moving the price, the same problem an execution desk solves."
2. **Data:** "I rebuilt the full limit order book from raw NASDAQ ITCH 5.0 messages: Add, Execute, Cancel, Delete, Replace."
3. **Simulator:** "On top of that, an event-driven execution simulator that inserts my child orders into the replayed book, models queue position, latency and fees, and decides when they fill."
4. **Agent:** "An RL agent that chooses when and how aggressively to trade. Reward: implementation shortfall vs arrival. Baselines: TWAP and VWAP."
5. **Result:** one number (bps vs TWAP, msgs/sec, days of data).
6. **Limits + next:** "Replay can't capture my own impact, so I'd add an impact model / test on more regimes."
7. **Tie-in:** "That's why this desk interests me: you do this for real flow."

In [ ]:
# My 90-second pitch (write it, then time yourself saying it):
"""

"""

### Deep-dive questions they may ask (answer skeletons)

**Q29. ⭐ How do you decide when *your* simulated order fills?**

<details><summary>Answer</summary>

It joins the back of the queue at its price level. It fills only after all quantity ahead of it is executed or cancelled (track "volume ahead"), or immediately if it crosses the spread (then it takes liquidity at the displayed levels). Add latency: your order arrives some microseconds/milliseconds after your decision. Be honest about simplifications, e.g. "I assume cancels ahead of me are spread evenly through the queue."
</details>

**Q30. ⭐ What are the RL agent's state, actions and reward?**

<details><summary>Answer</summary>

*Fill with your real design.* Typical: **state** = time left, inventory left, spread, book imbalance, recent volatility, queue position. **Actions** = do nothing / post at best / post one tick inside / cross the spread, or a participation rate. **Reward** = negative implementation shortfall (with a penalty for unfinished shares at the deadline). Compare against TWAP/VWAP on held-out days.
</details>

**Q31. ⭐ How do you know the agent isn't overfitting?**

<details><summary>Answer</summary>

Train and test on **different days** (chronological split, no shuffling), test across different regimes (volatile vs quiet days, different stocks), compare to simple baselines, look at the *distribution* of costs not just the mean, and check behaviour makes sense (e.g. it trades faster when time is running out).
</details>

**Q32. ⭐ Why Rust/Python? What was the hardest bug?**

<details><summary>Answer</summary>

*Fill in.* Template: "Parsing and book updates were the bottleneck in Python, so I moved them to Rust: X× faster. The hardest bug was ___ (e.g. Replace messages: the new order gets a new id and loses priority; or an off-by-one in big-endian timestamp parsing). I found it by ___ (checking an invariant like best bid < best ask after every message, or comparing against reference trade prints)."
</details>

**Q33. How would this change for a real desk?**

<details><summary>Answer</summary>

Real-time feeds instead of files; prices as integer ticks; C++/Java on the hot path; risk checks before every order; multiple venues and routing; FIX connectivity; monitoring and kill switch; and live A/B testing against existing algos instead of only back-tests.
</details>

In [ ]:
# My numbers (msgs/sec, # days/stocks, bps vs TWAP, hardest bug + how I found it):
"""

"""

---
# 10 · Behavioral + "why" answers
Keep each answer ~45 seconds. Concrete beats generic.

**Q34. ⭐ Tell me about yourself.**

<details><summary>Answer</summary>

"I'm a third-year Computer Engineering student at U of T, currently a software engineering co-op at TD on the e-signature team, where I build AI automation. Outside work, my main project is an execution simulator built on a limit order book I reconstruct from raw NASDAQ ITCH data, with an RL agent that learns how to execute large orders. Execution is the part of markets I find most interesting, which is why this role on the EES desk stood out to me."
</details>

**Q35. ⭐ Why this desk / why execution?**

<details><summary>Answer</summary>

"Execution is where engineering decisions show up directly in client outcomes: a better router or schedule is measurable in basis points. My project made me curious about exactly the questions you work on: when to post vs cross, how to split across venues, and how to measure quality with TCA. I'd love to see how that works with real flow and real constraints."
</details>

**Q36. ⭐ Why Scotiabank?**

<details><summary>Answer</summary>

Make it specific: "Scotiabank is one of Canada's largest banks with a major presence in Canadian equity markets, and its markets business spans the Americas. Working on a desk that executes for institutional clients in Canada's fragmented market is a great place to learn market structure from the inside. And it's in Toronto, where I'd like to build my career." *Swap in anything specific you learn from the recruiter or the job posting.*
</details>

**Q37. ⭐ Why trading developer rather than pure SWE or quant?**

<details><summary>Answer</summary>

"I want to be close to the business: building systems traders use every day, where performance and correctness matter, and the feedback is fast. I enjoy the engineering side most, but I like working with quantitative ideas, so a quant/trading developer role sits right where I want to be."
</details>

**Q38. ⭐ Tell me about a hard bug / a time you were stuck.**

<details><summary>Answer</summary>

STAR: **Situation** (project, stakes) → **Task** (what you had to fix) → **Action** (how you narrowed it down: invariants, logging, a minimal reproduction, a tool like a debugger or comparison against reference data) → **Result** (fixed, what changed, what you learned). Pick one from the LOB project or TD.
</details>

**Q39. A time you worked under time pressure / handled feedback / disagreed.**

<details><summary>Answer</summary>

Have **one** TD story ready (e.g. your demo on Oct 6: scope, deadline, what you cut, how it went) and one team story (AURA, 22 people). Use STAR, end with a result and a lesson.
</details>

**Q40. ⭐ What's your weakness? / What didn't go well recently?**

<details><summary>Answer</summary>

A real one, with what you're doing about it: "In a recent interview I knew the material but explained concepts less clearly than I wanted under pressure, so I've been practising answering technical questions out loud in a set structure." Honest, shows growth, and true.
</details>

**Q41. ⭐ Do you have experience with C++? Java?**

<details><summary>Answer</summary>

Be honest about level: what you built, what you've read (A Tour of C++, Effective Modern C++), the concepts you're comfortable with (RAII, smart pointers, move semantics, STL), and that you pick up languages fast (Rust in your project). Never overclaim; they will probe.
</details>

---
# 11 · Questions to ask them (pick 2–3)

- Which algorithms and venues make up most of the desk's flow, and where do clients push you hardest?
- How do developers work with the traders and quants day to day? Do devs sit on the desk?
- What does the stack look like (C++ / Java / Python), and where does latency matter most for you?
- How do you test and roll out a change to an algo or the router?
- What would a co-op/intern own in their first months, and what does success look like?
- How is TCA used: for clients, or to improve the algos internally?

---
# 12 · Mock list: rapid fire with a timer
Set a 60-second timer per question. Answer out loud, no notes. Mark ⭐ the ones you stumble on and redo them.

1. Walk me through your execution simulator. *(90 s)*
2. What's the difference between a market and a limit order? Give 3 other order types.
3. Explain price-time priority.
4. What is VWAP? How does a VWAP algo build its schedule?
5. TWAP vs POV: when would you use each?
6. Why split a large order? What's the trade-off between trading fast and slow?
7. What is implementation shortfall? How do you measure slippage in bps?
8. What does a smart order router consider?
9. What's the Order Protection Rule?
10. What's FIX? What are 35=D and 35=8? What happens on a sequence gap?
11. What pre-trade risk checks would you build? What's a kill switch?
12. Design an order book: data structures and complexity. *(then code it on paper)*
13. Why integer prices?
14. What happens when you type a URL in a browser?
15. What is Docker? What did people do before it?
16. Generators: what does `yield` do?
17. Is time complexity absolute?
18. What is recursion? What's the risk?
19. Smart pointers: `unique_ptr` vs `shared_ptr` vs `weak_ptr`.
20. Process vs thread. What is a race condition?
21. TCP vs UDP, and why market data uses UDP multicast.
22. Why Scotiabank? Why this desk? Why trading developer?
23. Tell me about a hard bug.
24. What's a weakness?

---
### Monday checklist
- [ ] 3 printed résumés, pen, notebook, ID, phone charged
- [ ] Business professional, comfortable shoes
- [ ] Leave so you're at 40 Temperance by **10:50**; ground-floor security → 5th-floor reception
- [ ] Re-read your 90-second pitch and the ⭐ answers on the way
- [ ] In the room: clarify before coding, think out loud, test your code with a small example, ask your 2–3 questions
- [ ] Afterwards: a short thank-you email the same day